## Sentiment Analysis

### Classify documents based on their sentiment using ML

We will download the movie review dataset from IMDB.

In [27]:
import tarfile
import pandas as pd
import numpy as np
import pyprind
import os
import sys
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# Unzip tar file 
with tarfile.open("data/aclImdb_v1.tar","r") as tar:
    tar.extractall(path="./data/")

In [10]:
basepath = 'data/aclImdb'

In [ ]:
## Reading reviews from both test and train dataset and appending them into one dataset and creating a label.
labels = {'pos': 1, 'neg': 0}
pbar = pyprind.ProgBar(50000, stream = sys.stdout)
df = pd.DataFrame()

for s in ('test', 'train'):
    for l in ('pos','neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding = 'utf-8') as infile:
                txt = infile.read()
            df = df.append([[txt, labels[l]]],
                           ignore_index = True)
            pbar.update()

df.columns = ['review', 'sentiment']

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:04:05


In [ ]:
## Reshuffle the index so we have a mix of both positive and negative review.
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))
# save the data.
df.to_csv('data/movie_data.csv', index = False, encoding='utf-8')

In [23]:
df = pd.read_csv('data/movie_data.csv', encoding='utf-8')
df.head()

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


## Bag-of-words model

Bag of word model allow us to represent text as numerical features vectors.

The idea behind bag-of-words model is :-
1. we create a vocabulary of unique tokens - for example, words - from the entire set of documents.
2. we construct  a feature vector from each document that contains the count of how often each word occurs in the particular document.

Since the unique word in a document is a small subset of bag-of-words vocabulary, most feature vector will be 0, hence we call them sparse.

### Transform words into feature vectors

We can use CountVectorizer that take an array of text and create bag-of-words model for us.

In [ ]:
## An Example
count = CountVectorizer()
docs = np.array(['The sun is shining',
        'The weather is sweet',
        'The sun is shining, the weather is sweet, and one and one is two'])
bag = count.fit_transform(docs) # construct vocab of bag-of-world model and transformed them into feature vector

In [ ]:
# bag-of-words vocabulary
count.vocabulary_

{'the': 6,
 'sun': 4,
 'is': 1,
 'shining': 3,
 'weather': 8,
 'sweet': 5,
 'and': 0,
 'one': 2,
 'two': 7}

The values for each word is their index. which means that 'and' has index 0, 'is' has index 1 and so on. So feature vector the value in index 0 will represent how many occurance of word 'and' has in a particulat text.

In [ ]:
## Sparse feature vectors
print(bag.toarray())

[[0 1 0 1 1 0 1 0 0]
 [0 1 0 0 0 1 1 0 1]
 [2 3 2 1 1 1 2 1 1]]
